In [9]:
#Importation de la librairie Pandas
import pandas as pd
import numpy as np



In [10]:
df_volaille_raw = pd.read_csv(
    "data/poultry.csv",
    sep=",",
    encoding="latin-1"
)

df_distance = pd.read_csv(
    "data/geo.csv",
    sep=";",
    encoding="latin-1"
)

df_logistique_raw = pd.read_csv(
    "data/Logistique.csv",
    sep=",",
    encoding="latin-1"
)

df_ouverture_raw = pd.read_csv(
    "data/ouverture.csv",
    sep=",",
    encoding="latin-1",
    skiprows=3 
)

df_bc_raw = pd.read_csv(
    "data/bc.csv",
    sep=",",
    encoding="latin-1"
)

df_inflation_raw = pd.read_csv(
    "data/inflation.csv",
    sep=",",
    encoding="latin-1",
    skiprows=3 
)

# Importation du fichier Politique.csv
df_politique_raw = pd.read_csv(
    "data/Politique.csv",
    sep=",",
    encoding="latin-1",
    on_bad_lines="skip"
)

# Importation du fichier population.csv
df_pop_raw = pd.read_csv(
    "data/population.csv",
    sep=",",
    encoding="latin-1",
)

df_fao = pd.read_csv(
    "data/chicken.csv",
    sep=",",
    encoding="latin-1"
)

df_pib_raw = pd.read_csv(
    "data/pib.csv",
    sep=",",
    encoding="latin-1",
    skiprows=3
)

In [11]:


print("Le tableau comporte {} observation(s)".format(df_volaille_raw.shape[0]))
print("Le tableau comporte {} colonne(s)".format(df_volaille_raw.shape[1]))

print("Le tableau comporte {} observation(s) ".format(df_pop_raw.shape[0]))
print("Le tableau comporte {} colonne(s)".format(df_pop_raw.shape[1]))

df_pop_raw.head(10)





print(df_pop_raw.columns.tolist())

Le tableau comporte 14619 observation(s)
Le tableau comporte 4 colonne(s)
Le tableau comporte 58824 observation(s) 
Le tableau comporte 4 colonne(s)
['Entity', 'Code', 'Year', 'Population']


In [12]:

from geopy.distance import geodesic

# ============================================================
# 1. DISTANCE ENTRE LES PAYS ET LA FRANCE
# ============================================================

df_distance = df_distance.copy()
df_distance = df_distance.rename(
    columns={
        "country": "Zone"
    }
)
df_distance["lat"] = pd.to_numeric(
    df_distance["lat"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

df_distance["lon"] = pd.to_numeric(
    df_distance["lon"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

df_distance = df_distance[
    [
        "Zone",
        "pays",
        "lat",
        "lon"
    ]
].copy()


paris = (48.8566, 2.3522)


def calcul_distance_france(row):

    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        return np.nan

    try:
        return geodesic(
            paris,
            (row["lat"], row["lon"])
        ).km

    except ValueError:
        return np.nan


df_distance["distance_france_km"] = df_distance.apply(
    calcul_distance_france,
    axis=1
)


# ============================================================
# 2. INFLATION
# ============================================================

df_inflation_raw = df_inflation_raw.rename(
    columns={
        "Country Name": "Zone",
        "2017": "inflation"
    }
)

df_inflation = df_inflation_raw[
    [
        "Zone",
        "inflation"
    ]
].copy()

df_inflation["inflation"] = pd.to_numeric(
    df_inflation["inflation"],
)


# ============================================================
# 3. POPULATION
# ============================================================

df_pop = df_pop_raw[
    [
        "Entity",
        "Code",
        "Year",
        "Population"
    ]
].copy()

df_pop = df_pop.rename(
    columns={
        "Entity": "Zone",
        "Code": "ISO3",
        "Year": "Année"
    }
)

df_pop["Année"] = pd.to_numeric(
    df_pop["Année"],
    errors="coerce"
)

df_pop["Population"] = pd.to_numeric(
    df_pop["Population"],
    errors="coerce"
)

# Garder uniquement l'année 2017
df_pop = df_pop[
    df_pop["Année"] == 2017
].copy()

# Garder une seule ligne par pays
df_pop = df_pop.drop_duplicates(
    subset=["ISO3"]
)

# Colonnes utiles pour les fusions
df_pop = df_pop[
    [
        "Zone",
        "ISO3",
        "Population"
    ]
].copy()


df_pop.sort_values(
    by="Population",
    ascending=False
).head(30)

a_supprimer = [
    "World",
    "Asia",
    "Africa",
    "Europe",
    "North America",
    "South America",
    "Oceania",
    "European Union",
    "High-income countries",
    "Upper-middle-income countries",
    "Lower-middle-income countries",
    "Low-income countries"
]

df_pop = df_pop[
    ~df_pop["Zone"].isin(a_supprimer)
]


# ============================================================
# 4. STABILITÉ POLITIQUE
# ============================================================

df_politique_raw = df_politique_raw.rename(
    columns={
        "Country Name": "Zone",
        "2017 [YR2017]": "Stabilité_Politique"
    }
)

df_politique = df_politique_raw[
    [
        "Zone",
        "Stabilité_Politique"
    ]
].copy()

df_politique["Stabilité_Politique"] = pd.to_numeric(
    df_politique["Stabilité_Politique"],
    errors="coerce"
)


# ============================================================
# 5. PIB PAR HABITANT
# ============================================================

df_pib = df_pib_raw[
    [
        "Country Name",
        "Country Code",
        "Indicator Name",
        "Indicator Code",
        "2017"
    ]
].copy()

df_pib = df_pib.rename(
    columns={
        "Country Name": "Zone",
        "2017": "PIB_habitant"
    }
)

df_pib["PIB_habitant"] = pd.to_numeric(
    df_pib["PIB_habitant"],
    errors="coerce"
)

# Garder uniquement les colonnes nécessaires
df_pib = df_pib[
    [
        "Zone",
        "PIB_habitant"
    ]
].copy()


# ============================================================
# 6. BALANCE COMMERCIALE
# ============================================================

balance_com_raw = df_bc_raw.pivot_table(
    index="Area",
    columns="Element",
    values="Value",
    aggfunc="sum"
)

balance_com_raw = balance_com_raw.fillna(0)

# Sécurité si une des deux colonnes n'existe pas
if "Export value" not in balance_com_raw.columns:
    balance_com_raw["Export value"] = 0

if "Import value" not in balance_com_raw.columns:
    balance_com_raw["Import value"] = 0

balance_com_raw["Balance_commerciale"] = (
    balance_com_raw["Export value"]
    - balance_com_raw["Import value"]
)

balance_com_raw = balance_com_raw.reset_index()

balance_com_raw = balance_com_raw.rename(
    columns={
        "Area": "Zone"
    }
)


balance_com = balance_com_raw[
    [
        "Zone",
        "Balance_commerciale"
    ]
].copy()


# ============================================================
# 7. OUVERTURE COMMERCIALE
# ============================================================

df_ouverture = df_ouverture_raw[
    [
        "Country Name",
        "2017"
    ]
].copy()

df_ouverture = df_ouverture.rename(
    columns={
        "Country Name": "Zone",
        "2017": "index_ouverture"
    }
)

df_ouverture["index_ouverture"] = pd.to_numeric(
    df_ouverture["index_ouverture"],
    errors="coerce"
)


# ============================================================
# 8. PERFORMANCE LOGISTIQUE
# ============================================================

df_logistique = df_logistique_raw[
    [
        "Country Name",
        "2018 [YR2018]"
    ]
].copy()

df_logistique = df_logistique.rename(
    columns={"Country Name": "Zone",
             "2018 [YR2018]": "index_log"
    }
)
df_logistique["index_log"] = pd.to_numeric(
    df_logistique["index_log"],
    errors="coerce"
)


# ============================================================
# 9. SUPPRESSION DES DOUBLONS
# ============================================================

balance_com = balance_com.drop_duplicates(
    subset=["Zone"]
)

df_pib = df_pib.drop_duplicates(
    subset=["Zone"]
)

df_politique = df_politique.drop_duplicates(
    subset=["Zone"]
)

df_inflation = df_inflation.drop_duplicates(
    subset=["Zone"]
)

df_ouverture = df_ouverture.drop_duplicates(
    subset=["Zone"]
)

df_logistique = df_logistique.drop_duplicates(
    subset=["Zone"]
)

df_distance = df_distance.drop_duplicates(
    subset=["Zone"]
)


# ============================================================
# 10. HARMONISATION DE LA COLONNE PAYS
# ============================================================

df_distance_test = df_distance.rename(
    columns={
        "country": "Zone"
    }
).copy()

df_distance_test = df_distance_test[
    [
        "Zone",
        "distance_france_km"
    ]
].copy()


balance_com_test = balance_com.rename(
    columns={
        "Area": "Zone"
    }
).copy()


df_ouverture_test = df_ouverture.rename(
    columns={
        "Country Name": "Zone",
        "2017": "index_ouverture"
    }
).copy()


df_logistique_test = df_logistique.rename(
    columns={
        "Country Name": "Zone",
        "2018 [YR2018]": "index_log"
    }
).copy()


df_volaille = df_volaille_raw[
    [
        "Entity",
        "Code",
        "Year",
        "Poultry meat production"
    ]
].copy()

df_volaille = df_volaille.rename(
    columns={
        "Entity": "Zone",
        "Code": "ISO3",
        "Year": "Année",
        "Poultry meat production": "disponibilite_volaille"
    }
)

df_volaille["Année"] = pd.to_numeric(
    df_volaille["Année"],
    errors="coerce"
)

df_volaille["disponibilite_volaille"] = pd.to_numeric(
    df_volaille["disponibilite_volaille"],
    errors="coerce"
)

# Garder uniquement 2017
df_volaille = df_volaille[
    df_volaille["Année"] == 2017
].copy()

# Supprimer les agrégats (World, Asia, etc.)
df_volaille = df_volaille.dropna(subset=["ISO3"])

df_volaille = df_volaille.drop_duplicates(
    subset=["Zone"]
)

# Colonnes utiles
df_volaille = df_volaille[
    [
        "Zone",
        "disponibilite_volaille"
    ]
]
# ============================================================
# 11. NOMBRE DE PAYS PRÉSENTS DANS CHAQUE FICHIER
# ============================================================

dataframes = {
    "Population": df_pop,
    "Inflation": df_inflation,
    "Stabilité politique": df_politique,
    "PIB par habitant": df_pib,
    "Balance commerciale": balance_com_test,
    "Ouverture commerciale": df_ouverture_test,
    "Logistique": df_logistique_test,
    "Distance": df_distance_test,
    "disponibilite_volaille": df_volaille
}

resultats = []

for nom, dataframe in dataframes.items():

    resultats.append({
        "Fichier": nom,
        "Nombre_pays": dataframe["Zone"].nunique(),
        "Zones_manquantes": dataframe["Zone"].isna().sum()
    })


comparaison_fichiers = pd.DataFrame(
    resultats
).sort_values(
    by="Nombre_pays",
    ascending=True
)

print("Nombre de pays dans chaque fichier :")

print(comparaison_fichiers)


# ============================================================
# 12. TEST DES FUSIONS SUCCESSIVES
# ============================================================

# L'inflation sert de base de départ
df_test = df_inflation.copy()

suivi = []

dataframes_a_fusionner = {
    "Population": df_pop,
    "Stabilité politique": df_politique,
    "PIB par habitant": df_pib,
    "Balance commerciale": balance_com_test,
    "Ouverture commerciale": df_ouverture_test,
    "Logistique": df_logistique_test,
    "Distance": df_distance_test,
    "disponibilite_volaille": df_volaille
}


for nom, dataframe in dataframes_a_fusionner.items():

    pays_avant = set(
        df_test["Zone"]
        .dropna()
        .unique()
    )

    df_test = df_test.merge(
        dataframe,
        on="Zone",
        how="inner",
        validate="one_to_one"
    )

    pays_apres = set(
        df_test["Zone"]
        .dropna()
        .unique()
    )

    pays_perdus = sorted(
        pays_avant - pays_apres
    )

    suivi.append({
        "Fusion": nom,
        "Pays_avant": len(pays_avant),
        "Pays_apres": len(pays_apres),
        "Pays_perdus": len(pays_perdus),
        "Liste_pays_perdus": pays_perdus
    })


suivi_fusions = pd.DataFrame(suivi)


print("\nSuivi des fusions :")

print(
    suivi_fusions[
        [
            "Fusion",
            "Pays_avant",
            "Pays_apres",
            "Pays_perdus"
        ]
    ].sort_values(
        by="Pays_perdus",
        ascending=False
    )
)


# ============================================================
# 13. AFFICHAGE DES PAYS PERDUS À CHAQUE FUSION
# ============================================================

for _, ligne in suivi_fusions.iterrows():

    print(f"\nFusion : {ligne['Fusion']}")
    print(f"Pays perdus : {ligne['Pays_perdus']}")

    if ligne["Pays_perdus"] > 0:
        print(ligne["Liste_pays_perdus"])


# ============================================================
# 14. RÉSULTAT FINAL
# ============================================================

print("\nDimensions finales :")
print(df_test.shape)

print("\nColonnes finales :")
print(df_test.columns.tolist())

print("\nAperçu :")
print(df_test.head())

print("\nValeurs manquantes :")
print(
    df_test.isna()
    .sum()
    .sort_values(ascending=False)
)

Nombre de pays dans chaque fichier :
                  Fichier  Nombre_pays  Zones_manquantes
6              Logistique          174                 1
4     Balance commerciale          187                 0
8  disponibilite_volaille          206                 0
2     Stabilité politique          218                 1
7                Distance          225                 0
0              Population          246                 0
5   Ouverture commerciale          265                 0
3        PIB par habitant          266                 0
1               Inflation          266                 0

Suivi des fusions :
                   Fusion  Pays_avant  Pays_apres  Pays_perdus
0              Population         266         185           81
3     Balance commerciale         177         150           27
5              Logistique         150         135           15
1     Stabilité politique         185         177            8
6                Distance         135         129        

In [13]:
df_final = df_pop[
    [
        "Zone",
        "ISO3",
        "Population"
    ]
].copy()


df_final = df_final.merge(
    df_volaille,
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    df_pib[["Zone", "PIB_habitant"]],
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    df_politique[["Zone", "Stabilité_Politique"]],
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    df_inflation[["Zone", "inflation"]],
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    df_ouverture[["Zone", "index_ouverture"]],
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    df_logistique[["Zone", "index_log"]],
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    balance_com[["Zone", "Balance_commerciale"]],
    on="Zone",
    how="left"
)

df_final = df_final.merge(
    df_distance[["Zone", "distance_france_km"]],
    on="Zone",
    how="left"
)


# Conversion numérique
colonnes_num = [
    "Population",
    "PIB_habitant",
    "Stabilité_Politique",
    "inflation",
    "index_ouverture",
    "index_log",
    "Balance_commerciale",
    "distance_france_km"
    "disponibilite_volaille"
]

In [14]:
print(df_final)

                  Zone    ISO3  Population  disponibilite_volaille  \
0          Afghanistan     AFG    35688941                27637.84   
1          Africa (UN)  UN_AFR  1282902113                     NaN   
2              Albania     ALB     2898245                14414.11   
3              Algeria     DZA    41689302               400005.80   
4       American Samoa     ASM       51605                     NaN   
..                 ...     ...         ...                     ...   
241  Wallis and Futuna     WLF       11945                     NaN   
242     Western Sahara     ESH      512851                     NaN   
243              Yemen     YEM    33090923               168628.27   
244             Zambia     ZMB    17441328                47420.58   
245           Zimbabwe     ZWE    14812484                92300.32   

     PIB_habitant  Stabilité_Politique  inflation  index_ouverture  index_log  \
0      525.469771            -1.442187   4.975952              NaN   1.790851 

In [15]:
df_final = df_final[df_final["Zone"] != "France"].copy()


In [16]:
df_final['Population_log'] = np.log1p(df_final['Population'])
df_final['PIB_habitant_log'] = np.log1p(df_final['PIB_habitant'])
df_final['Stabilité_Politique_log'] = np.log1p(df_final["Stabilité_Politique"])
df_final['inflation_log'] = np.log1p(df_final["inflation"])
df_final["distance_france_log"] = np.log1p(df_final["distance_france_km"])
df_final["index_ouverture_log"] = np.log1p(df_final["index_ouverture"])
df_final["disponibilite_volaille_log"] = np.log1p(df_final["disponibilite_volaille"])



from sklearn.preprocessing import StandardScaler

variables_acp = [
    "Population_log",
    "PIB_habitant_log",
    "Stabilité_Politique",
    "inflation",
    "index_ouverture",
    "index_log",
    "Balance_commerciale",
    "distance_france_log",
    "disponibilite_volaille"
]

X = df_final[variables_acp].copy()

X_scaled = StandardScaler().fit_transform(X)

X = X.apply(pd.to_numeric, errors="coerce")

# Remplacer les valeurs infinies par NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Afficher les valeurs manquantes
print("Valeurs manquantes avant nettoyage :")
print(X.isna().sum().sort_values(ascending=False))


X = X.dropna()





c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Valeurs manquantes avant nettoyage :
index_log                 106
index_ouverture            86
inflation                  85
Balance_commerciale        82
Stabilité_Politique        66
PIB_habitant_log           65
disponibilite_volaille     51
distance_france_log        50
Population_log              0
dtype: int64


In [17]:
df_final.to_csv(
    "data/df_final_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_final.to_excel(
    "data/df_final_clean.xlsx",
    index=False
)